Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!
No optimization was used.

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

import optuna

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
)

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'default'
add_smote = True
is_smotenc = True

smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

# Logistic Regression

In [5]:
from sklearn.metrics import (
    make_scorer,
    f1_score,
)
import optuna
from functools import partial

estimator_params = {
    "fit_intercept": False,
}

estimator = LogisticRegression(
    **estimator_params
)

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    scoring_metrics={"f1_macro": make_scorer(f1_score, average="macro"),}
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

In [6]:
class OptunaOptimizer:
    def __init__(
        self, 
        objective:callable,
        study_name:str,
        direction:str="maximize",
    ):
        """Initialize the optimizer.

        Args:
            objective: The objective function to optimize.
            study_name: The name of the study.
            direction: The direction of optimization (maximize or minimize).
        """
        self.objective = objective
        self.study_name = study_name
        self.direction = direction
        
        self.study = optuna.create_study(
            study_name=study_name,
            direction=direction,
        )
        
    def optimize(self, n_trials:int):
        """Optimize the objective function.

        Args:
            n_trials: The number of trials to run for optimization.

        Returns:
            The study object containing the optimization results.
        """
        self.study.optimize(
            self.objective,
            n_trials=n_trials,
        )
        
def update_estimator_params(
    ml_pipe:MLPipeline,
    suggested_params:dict,
) -> dict:
    """Get the estimator parameters from the pipeline parameters.

    Args:
        ml_pipe: An instance of MLPipeline used for model training and evaluation.
        suggested_params: A dictionary containing the suggested hyperparameters.
    
    Returns:
        A dictionary containing the estimator parameters.
    """
    estimator_params = ml_pipe._pipeline_params['estimator_params']
    estimator_params.update(suggested_params)
    return estimator_params

In [7]:
def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    """Objective function for logistic regression optimization using Optuna.

    Args:
        trial: An Optuna trial object used to suggest hyperparameters.
        ml_pipe: An instance of MLPipeline used for model training and evaluation.

    Returns:
        The score of the model based on the specified evaluation metric.
    """
    
    categorical_features = ('wettability', 'inclination')
    
    random_state = ml_pipe._pipeline_params['random_state']
    
    # sample params
    C = trial.suggest_loguniform('C', 1e-4, 1e3)
    
    # SMOTE params
    add_smote = trial.suggest_categorical('add_smote', [True, False])
    if add_smote:
        is_smotenc = trial.suggest_categorical('is_smotenc', [True, False])
        smote_params = {
            'sampling_strategy': trial.suggest_categorical(
                'sampling_strategy', [0.6, 0.7, 0.8, 0.9, 1.0]
            ),
            'k_neighbors': trial.suggest_int('k_neighbors', 3, 10),
            'random_state': random_state,
        }
    else:
        is_smotenc = False
        smote_params = None
    if is_smotenc:
        wettability_cat = trial.suggest_categorical('wettability_cat', [True, False])
        if wettability_cat:
            inclination_cat = trial.suggest_categorical('inclination_cat', [True, False])
        else:
            inclination_cat = True
        
        
        mask = [wettability_cat, inclination_cat]
        
        masked_features = [
            feature for feature, mask_value in zip(categorical_features, mask) 
            if mask_value
        ]
        smote_params = {
            **smote_params,
            'categorical_features': masked_features,
        }

    suggested_params = {
        "C": C,
    }
    
    estimator_params = update_estimator_params(ml_pipe, suggested_params)

    estimator = LogisticRegression(
        **estimator_params,
        random_state=random_state,
    )

    score = ml_pipe.step(
        estimator=estimator,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
    )
    
    return score



opt = OptunaOptimizer(
    objective=partial(logreg_objective, ml_pipe=ml_pipe),
    study_name="logreg_study",
    direction="maximize",
)

opt.optimize(n_trials=200)

best_params = opt.study.best_trial.params
display(best_params)
# estimator_params = update_estimator_params(ml_pipe, best_params)

# estimator = LogisticRegression(
#     **estimator_params,
#     random_state=ml_pipe._pipeline_params['random_state'],
# )

[I 2025-04-11 00:02:50,034] A new study created in memory with name: logreg_study
[I 2025-04-11 00:02:50,105] Trial 0 finished with value: 0.7754010695187166 and parameters: {'C': 7.659422407680541, 'add_smote': False}. Best is trial 0 with value: 0.7754010695187166.
[I 2025-04-11 00:02:50,166] Trial 1 finished with value: 0.7730205278592375 and parameters: {'C': 0.3986291498738802, 'add_smote': False}. Best is trial 0 with value: 0.7754010695187166.
[I 2025-04-11 00:02:50,237] Trial 2 finished with value: 0.7787307032590051 and parameters: {'C': 0.0030108676374517194, 'add_smote': True, 'is_smotenc': False, 'sampling_strategy': 0.6, 'k_neighbors': 5}. Best is trial 2 with value: 0.7787307032590051.
[I 2025-04-11 00:02:50,569] Trial 3 finished with value: 0.800925925925926 and parameters: {'C': 17.34448264461045, 'add_smote': True, 'is_smotenc': True, 'sampling_strategy': 0.9, 'k_neighbors': 4, 'wettability_cat': True, 'inclination_cat': False}. Best is trial 3 with value: 0.8009259259

{'C': 1.6850403782968544,
 'add_smote': True,
 'is_smotenc': False,
 'sampling_strategy': 0.7,
 'k_neighbors': 4}

TODO: Предварительно проверить метрики SMOTE и SMOTENC и исключить что-то. Например, SMOTENC.

In [9]:
estimator_params

{'fit_intercept': False, 'C': 46.368205497627464}

In [9]:
ml_pipe.metric_results

[]

In [7]:
estimator = LogisticRegression(
    fit_intercept=False,
)

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smotenc_default
holdout_test_f1_macro,0.755981
holdout_test_accuracy_balanced,0.758102
holdout_test_roc_auc,0.878858
holdout_test_f1,0.821053
holdout_test_accuracy,0.773333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.816667
cv_test_roc_auc_median,0.873065


Model saved in ../results/models_modelling4/LogisticRegression_splashing_smotenc_default


In [8]:
estimator = LogisticRegression(
    fit_intercept=False,
)

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smotenc_de...
holdout_test_f1_macro,0.802991
holdout_test_accuracy_balanced,0.85
holdout_test_roc_auc,0.958182
holdout_test_f1,0.734694
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.813161
cv_test_accuracy_balanced_median,0.864469
cv_test_roc_auc_median,0.93956


Model saved in ../results/models_modelling4/LogisticRegression_no_fragmentation_smotenc_default


# KNClassifier

In [9]:
estimator = KNeighborsClassifier()

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smotenc_default
holdout_test_f1_macro,0.829027
holdout_test_accuracy_balanced,0.834491
holdout_test_roc_auc,0.887346
holdout_test_f1,0.87234
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.844575
cv_test_accuracy_balanced_median,0.862229
cv_test_roc_auc_median,0.917957


Model saved in ../results/models_modelling4/KNeighborsClassifier_splashing_smotenc_default


In [10]:
estimator = KNeighborsClassifier()

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smotenc_...
holdout_test_f1_macro,0.843227
holdout_test_accuracy_balanced,0.877273
holdout_test_roc_auc,0.954545
holdout_test_f1,0.782609
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.881326
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.927656


Model saved in ../results/models_modelling4/KNeighborsClassifier_no_fragmentation_smotenc_default


# SVC

In [11]:
estimator = SVC(probability=True)

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smotenc_default
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.877315
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.82623
cv_test_accuracy_balanced_median,0.856037
cv_test_roc_auc_median,0.933437


Model saved in ../results/models_modelling4/SVC_splashing_smotenc_default


In [12]:
estimator = SVC(probability=True)

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smotenc_default
holdout_test_f1_macro,0.860566
holdout_test_accuracy_balanced,0.902273
holdout_test_roc_auc,0.948182
holdout_test_f1,0.808511
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.881326
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.96337


Model saved in ../results/models_modelling4/SVC_no_fragmentation_smotenc_default


# CatBoost

In [13]:
estimator = CatBoostClassifier(verbose=False)

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_smotenc_default
holdout_test_f1_macro,0.868703
holdout_test_accuracy_balanced,0.865741
holdout_test_roc_auc,0.948302
holdout_test_f1,0.907216
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.956656


Model saved in ../results/models_modelling4/CatBoostClassifier_splashing_smotenc_default


In [14]:
estimator = CatBoostClassifier(verbose=False)

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smotenc_de...
holdout_test_f1_macro,0.916089
holdout_test_accuracy_balanced,0.922727
holdout_test_roc_auc,0.984545
holdout_test_f1,0.878049
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.881326
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.97265


Model saved in ../results/models_modelling4/CatBoostClassifier_no_fragmentation_smotenc_default


# XGBoost

In [15]:
estimator = XGBClassifier()

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smotenc_default
holdout_test_f1_macro,0.88
holdout_test_accuracy_balanced,0.868056
holdout_test_roc_auc,0.932485
holdout_test_f1,0.92
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.946032


Model saved in ../results/models_modelling4/XGBClassifier_splashing_smotenc_default


In [16]:
estimator = XGBClassifier()

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smotenc_default
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.988182
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.877289
cv_test_roc_auc_median,0.967033


Model saved in ../results/models_modelling4/XGBClassifier_no_fragmentation_smotenc_default


# AdaBoost

In [17]:
estimator = AdaBoostClassifier()

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smotenc_default
holdout_test_f1_macro,0.751994
holdout_test_accuracy_balanced,0.75
holdout_test_roc_auc,0.830633
holdout_test_f1,0.824742
holdout_test_accuracy,0.773333
cv_test_f1_macro_median,0.799242
cv_test_accuracy_balanced_median,0.806502
cv_test_roc_auc_median,0.902477


Model saved in ../results/models_modelling4/AdaBoostClassifier_splashing_smotenc_default


In [18]:
estimator = AdaBoostClassifier()

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smotenc_de...
holdout_test_f1_macro,0.839194
holdout_test_accuracy_balanced,0.861364
holdout_test_roc_auc,0.951364
holdout_test_f1,0.772727
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.826067
cv_test_accuracy_balanced_median,0.823077
cv_test_roc_auc_median,0.930403


Model saved in ../results/models_modelling4/AdaBoostClassifier_no_fragmentation_smotenc_default


# Random Forest

In [19]:
estimator = RandomForestClassifier()

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smotenc_default
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.937886
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.858018
cv_test_accuracy_balanced_median,0.873016
cv_test_roc_auc_median,0.945046


Model saved in ../results/models_modelling4/RandomForestClassifier_splashing_smotenc_default


In [20]:
estimator = RandomForestClassifier()

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smoten...
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.99
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.954701


Model saved in ../results/models_modelling4/RandomForestClassifier_no_fragmentation_smotenc_default


# LightGBM

In [21]:
estimator = LGBMClassifier()

ml_pipe = MLPipeline(
    target='splashing',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[LightGBM] [Info] Number of positive: 205, number of negative: 205
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000323 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 627
[LightGBM] [Info] Number of data points in the train set: 410, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,0
target,splashing
model,LGBMClassifier_splashing_smotenc_default
holdout_test_f1_macro,0.852826
holdout_test_accuracy_balanced,0.847222
holdout_test_roc_auc,0.957562
holdout_test_f1,0.897959
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.948916


Model saved in ../results/models_modelling4/LGBMClassifier_splashing_smotenc_default


In [22]:
estimator = LGBMClassifier()

ml_pipe = MLPipeline(
    target='no_fragmentation',
    estimator=estimator,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
)

ml_pipe.run()

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[LightGBM] [Info] Number of positive: 234, number of negative: 234
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000350 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 746
[LightGBM] [Info] Number of data points in the train set: 468, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smotenc_default
holdout_test_f1_macro,0.916089
holdout_test_accuracy_balanced,0.922727
holdout_test_roc_auc,0.986364
holdout_test_f1,0.878049
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.886022
cv_test_accuracy_balanced_median,0.90293
cv_test_roc_auc_median,0.974359


Model saved in ../results/models_modelling4/LGBMClassifier_no_fragmentation_smotenc_default
